[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tides/tides.ipynb)

# Tides

This example simulates a star cluster falling past a soft gravitational mass and compares the cluster's deformation against the first-order tidal prediction along its centre. Because gravity pulls harder on the near side of the cluster than on the far side, a passing body stretches along the line toward the source mass and compresses across it, drawing the stars out into an elongated stream. Each gravitating source is modelled as a Plummer sphere whose density stays finite inside its core and falls off smoothly into an inverse-square pull at large distance.

At each point, the tidal map sends an intrinsic displacement to the extrinsic plane reading the change in gravity (`Plane <- Point`), combining an isotropic squeeze `-pull_strength * Point.dual_inverse()` with a radial stretch along `source_planes * (source_planes & Point)`. Tracing `(Plane * tides(Point)).trace(1, 2)` forms the geometric derivative of gravity directly from the tidal map, whose bivector curl vanishes by symmetry without any metric and whose scalar divergence recovers minus four pi times the Plummer density via Poisson's equation. Along the cluster's guiding centre, dualizing `tides(deformation).dual()` turns that plane reading into the tidal acceleration of the linear deformation map, pushing an initial round spread `cluster_radius**2 * Plane.dual()` forward with its adjugate `deformation(start_spread(deformation.adjugate()))` so that `predicted_spread.solve(offsets) & offsets == 1` marks the deformed rim.

In Cartesian tensor and matrix notation, the field equations split into the scalar divergence $\nabla\cdot\mathbf{g} = \sum_i \partial_i g_i = -4\pi G\rho$ and the Levi-Civita curl $(\nabla\times\mathbf{g})_i = \varepsilon_{ijk}\partial_j g_k = 0$, while the tidal matrix $T_{ij} = -\partial_i\partial_j\Phi = \sum_a G M_a \bigl(3\,\Delta x_{a,i}\Delta x_{a,j} - (|\Delta\mathbf{x}_a|^2 + b^2)\delta_{ij}\bigr)(|\Delta\mathbf{x}_a|^2 + b^2)^{-5/2}$ acts on coordinate column vectors. Along the guiding trajectory $\mathbf{x}_c(t)$, the $3\times 3$ Jacobian matrix $\mathbf{D}$ integrates $\ddot{\mathbf{D}} = \mathbf{T}(\mathbf{x}_c)\,\mathbf{D}$ and maps an initial covariance $r_0^2\mathbf{I}$ by matrix transpose to $\boldsymbol{\Sigma} = r_0^2\mathbf{D}\mathbf{D}^{\mathsf{T}}$, whose inverse quadratic form $\Delta\mathbf{x}^{\mathsf{T}}\boldsymbol{\Sigma}^{-1}\Delta\mathbf{x} = 1$ defines the deformed boundary.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext
from numga.algebras import VGA3D as ga

from examples.mechanics.tides import render
from examples.util.animation import show_states

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
# Positions, displacements, velocities and accelerations are intrinsic antivectors; planes are extrinsic vectors.
Point = ga.gatype.antivector()                              # Point-
Plane = ga.gatype.vector()                                  # Plane+
Even = ga.gatype.even()                                     # Even+
Tidal = ga.gatype((Plane, Point))                           # Plane+ <- Point-
Deformation = ga.gatype((Point, Point))                     # Point- <- Point-
Spread = ga.gatype((Point, Plane))                          # Point- <- Plane+

## Soft masses, tidal map, and Poisson's equation

Beyond its core radius a Plummer sphere pulls like a point mass, while inside the core the pull tapers smoothly to zero at the centre. Moving slightly away from a source weakens its pull in every direction, whereas moving along the line toward the source strengthens the radial pull, so each mass contributes an isotropic squeeze and a radial stretch to the tidal map. Multiplying each output plane of the tidal map by the plane reading its input step and taking the trace forms the geometric derivative of gravity, whose bivector curl vanishes identically without using the metric and whose scalar part recovers minus four pi times the Plummer density.

In [ ]:
mass_values = np.array([1.0])
mass_core = 0.3
half_width, half_height = 3.6, 2.4
n_columns, n_rows = 360, 240

In [ ]:
def gravitational_acceleration(
    points: Point, centres: Point, masses: np.ndarray, core_radius: float
) -> Point:
    """Plummer spheres at `centres` pull each sample in `points` toward them (`[...] Point-`)."""
    source_offsets = centres - points[..., None]                             # [...] Point-
    softened_squares = source_offsets.scalar_norm_squared() + core_radius**2  # [...] Scalar+
    pull_strength = masses / (softened_squares * softened_squares.square_root())  # [...] Scalar+
    return (pull_strength * source_offsets).sum(axis=-1)                     # [...] Point-


def tidal_map(
    points: Point, centres: Point, masses: np.ndarray, core_radius: float
) -> Tidal:
    """Each Plummer sphere combines an isotropic squeeze with a radial stretch into the tidal map at `points` (`[...] Plane+ <- Point-`)."""
    source_offsets = centres - points[..., None]                             # [...] Point-
    softened_squares = source_offsets.scalar_norm_squared() + core_radius**2  # [...] Scalar+
    pull_strength = masses / (softened_squares * softened_squares.square_root())  # [...] Scalar+
    pull_slope = -1.5 * pull_strength / softened_squares                     # [...] Scalar+
    # Each radial plane pairs with a displacement to read its component toward that mass.
    source_planes = source_offsets.dual_inverse()                            # [...] Plane+
    squeeze_map = -pull_strength * Point.dual_inverse()                      # [...] Plane+ <- Point-
    stretch_map = -2 * pull_slope * source_planes * (source_planes & Point)  # [...] Plane+ <- Point-
    return (squeeze_map + stretch_map).sum(axis=-1)                          # [...] Plane+ <- Point-

In [ ]:
def gravity_derivative(tides: Tidal) -> Even:
    """Tracing `Plane * tides(Point)` combines the scalar divergence and bivector curl of gravity (`[...] Even+`)."""
    # Tracing pairs each factor of Plane with the displacement argument of tides.
    return (Plane * tides(Point)).trace(1, 2)                                # [...] Even+


def sample_grid(
    half_width: float, half_height: float, n_columns: int, n_rows: int
) -> Point:
    """Cell centres span a `2 * half_width` by `2 * half_height` rectangle in the xy-plane (`[n_rows, n_columns] Point-`)."""
    x_offsets = (np.arange(n_columns) + 0.5) / n_columns * 2 * half_width - half_width
    y_offsets = (np.arange(n_rows) + 0.5) / n_rows * 2 * half_height - half_height
    return (mv.x * x_offsets[None, :] + mv.y * y_offsets[:, None]).dual()    # [n_rows, n_columns] Point-


mass_centres = mv.vector([[0, 0, 0]]).dual()                                 # [1] Point-
points = sample_grid(half_width, half_height, n_columns, n_rows)             # [n_rows, n_columns] Point-
field_derivative = gravity_derivative(
    tidal_map(points, mass_centres, mass_values, mass_core)
)                                                                            # [n_rows, n_columns] Even+
mass_density = -field_derivative.select[0] / (4 * np.pi)                     # [n_rows, n_columns] Scalar+
print(
    "Tidal map one unit from the mass along x:\n",
    tidal_map(mv.x.dual(), mass_centres, mass_values, mass_core).coefficient_table(),
)

## Tidal deformation of a falling star cluster

Each star in the disc falls independently under the Plummer mass by velocity Verlet integration. Along the guiding centre trajectory, dualizing `tides(deformation).dual()` turns the tidal map's plane reading into the acceleration of the linear deformation map from initial offsets to current offsets. Starting from a round disc of spread `cluster_radius**2 * Plane.dual()`, the deformation pushes offsets forward while its adjugate pulls reading planes back through `deformation(start_spread(deformation.adjugate()))`, predicting both the stars' second-moment spread and the cluster's elliptical rim from the centre trajectory alone. Integrating the flyby and rendering its frames takes about ten seconds.

In [ ]:
n_stars = 300
cluster_radius = 0.25
seed = 1
start_position, start_velocity = [-3.0, 0.5, 0.0], [0.5, 0.0, 0.0]
dt = 0.005
n_substeps = 8
n_frames = 150
frame_duration_ms = 40

In [ ]:
def star_step(
    positions: Point,
    velocities: Point,
    centres: Point,
    masses: np.ndarray,
    core_radius: float,
    dt: float,
) -> tuple[Point, Point]:
    """Velocity Verlet advances star positions and velocities across one time step `dt` (`[...] Point-`, `[...] Point-`)."""
    start_acceleration = gravitational_acceleration(positions, centres, masses, core_radius)  # [...] Point-
    next_positions = positions + velocities * dt + start_acceleration * (dt**2 / 2)           # [...] Point-
    end_acceleration = gravitational_acceleration(next_positions, centres, masses, core_radius)  # [...] Point-
    next_velocities = velocities + (start_acceleration + end_acceleration) * (dt / 2)         # [...] Point-
    return next_positions, next_velocities


def deformation_step(
    centre: Point,
    velocity: Point,
    deformation: Deformation,
    deformation_rate: Deformation,
    centres: Point,
    masses: np.ndarray,
    core_radius: float,
    dt: float,
) -> tuple[Point, Point, Deformation, Deformation]:
    """Velocity Verlet advances the guiding centre and its tidal deformation across `dt` (`[] Point-`, `[] Point- <- Point-`)."""
    # Dualization turns the tidal map's plane reading into an intrinsic acceleration offset.
    start_acceleration = gravitational_acceleration(centre, centres, masses, core_radius)     # [] Point-
    start_tide = tidal_map(centre, centres, masses, core_radius)(deformation).dual()          # [] Point- <- Point-
    next_centre = centre + velocity * dt + start_acceleration * (dt**2 / 2)                   # [] Point-
    next_deformation = deformation + deformation_rate * dt + start_tide * (dt**2 / 2)         # [] Point- <- Point-
    end_acceleration = gravitational_acceleration(next_centre, centres, masses, core_radius)  # [] Point-
    end_tide = tidal_map(next_centre, centres, masses, core_radius)(next_deformation).dual()  # [] Point- <- Point-
    next_velocity = velocity + (start_acceleration + end_acceleration) * (dt / 2)             # [] Point-
    next_rate = deformation_rate + (start_tide + end_tide) * (dt / 2)                         # [] Point- <- Point-
    return next_centre, next_velocity, next_deformation, next_rate

In [ ]:
def cluster_moment(positions: Point, centre: Point) -> Spread:
    """Star offsets around `centre` form the empirical second-moment map (`[...] Point- <- Plane+`)."""
    offsets = positions - centre[..., None]                                  # [...] Point-
    return (offsets * (Plane & offsets)).mean(axis=-1)                       # [...] Point- <- Plane+


def flyby_states(
    positions: Point,
    velocities: Point,
    guiding_centre: Point,
    guiding_velocity: Point,
    centres: Point,
    masses: np.ndarray,
    core_radius: float,
    cluster_radius: float,
    dt: float,
    n_substeps: int,
    n_frames: int,
) -> Iterator[tuple[Point, Point, Spread]]:
    """Yields `(positions, star_centre, predicted_spread)` across `n_frames` frames separated by `n_substeps` steps."""
    # Roundness enters through Plane.dual(), which maps each unit plane to the offset of equal length across it.
    start_spread = cluster_radius**2 * Plane.dual()                          # [] Point- <- Plane+
    deformation = Point                                                      # [] Point- <- Point-
    deformation_rate = 0 * Point                                             # [] Point- <- Point-
    for _ in range(n_frames):
        star_centre = positions.mean(axis=-1)                                # [] Point-
        predicted_spread = deformation(start_spread(deformation.adjugate()))  # [] Point- <- Plane+
        yield positions, star_centre, predicted_spread
        for _ in range(n_substeps):
            positions, velocities = star_step(
                positions, velocities, centres, masses, core_radius, dt
            )
            guiding_centre, guiding_velocity, deformation, deformation_rate = deformation_step(
                guiding_centre, guiding_velocity, deformation, deformation_rate, centres, masses, core_radius, dt
            )


rng = np.random.default_rng(seed)
# Radii grow as the square root so stars cover the disc uniformly under random rotor turns.
radii = cluster_radius * np.sqrt(rng.uniform(size=n_stars))
turns = (mv.xy * (-np.pi * rng.uniform(size=n_stars))).exp()                 # [n_stars] Rotor+
offsets = turns >> (mv.x.dual() * radii)                                     # [n_stars] Point-
guiding_centre = mv.vector(start_position).dual()                            # [] Point-
guiding_velocity = mv.vector(start_velocity).dual()                          # [] Point-
star_positions = guiding_centre + offsets                                    # [n_stars] Point-
star_velocities = guiding_velocity.broadcast_to(offsets.shape)               # [n_stars] Point-

states = show_states(
    flyby_states(
        star_positions,
        star_velocities,
        guiding_centre,
        guiding_velocity,
        mass_centres,
        mass_values,
        mass_core,
        cluster_radius,
        dt,
        n_substeps,
        n_frames,
    ),
    render.flyby_frames,
    points,
    mass_density,
    name="tides_flyby",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Logarithmic blue shading in the left panel shows the Plummer mass density recovered from the geometric derivative of gravity, with dark markers tracking the individual stars as they swing past the core. The zoomed right panel centres on the stars' centre of mass and overlays the orange level curve `predicted_spread.solve(offsets) & offsets == 1`, which elongates and tilts along the orbital stream in close agreement with the star cloud until nonlinear core passage bends the stream tips slightly.

In [ ]:
# checks
# Gravity's geometric derivative has zero curl and recovers minus four pi times the Plummer density.
source_offsets = mass_centres - points[..., None]                            # [n_rows, n_columns, 1] Point-
softened_squares = source_offsets.scalar_norm_squared() + mass_core**2       # [n_rows, n_columns, 1] Scalar+
plummer_density = (
    3 * mass_values * mass_core**2 / (4 * np.pi) / (softened_squares * softened_squares * softened_squares.square_root())
).sum(axis=-1)                                                               # [n_rows, n_columns] Scalar+
assert (field_derivative.select[2].norm() < 1e-10).all()
assert ((mass_density - plummer_density).abs() < 1e-10).all()

# A uniform disc's second moment is a quarter of its squared radius, carried forward by the tidal deformation.
n_sample_planes = 8
sample_planes = (mv.xy * (-np.pi * np.arange(n_sample_planes) / n_sample_planes)).exp() >> mv.x  # [n_sample_planes] Plane+
for state_positions, state_centre, state_spread in states:
    measured_moment = sample_planes & cluster_moment(state_positions, state_centre)(sample_planes)  # [n_sample_planes] Scalar+
    predicted_moment = sample_planes & (state_spread / 4)(sample_planes)     # [n_sample_planes] Scalar+
    assert ((measured_moment - predicted_moment).abs() <= 0.3 * predicted_moment.max()).all()